# Predicting Heart Disease — Machine Learning 

## 1. Project overview
Goal: build and evaluate supervised learning models to predict presence of heart disease using the UCI Heart Disease dataset. We'll cover EDA, preprocessing, modeling, evaluation, and interpretation.

Key questions:
- Which features are most predictive for heart disease?
- How well can a model discriminate between patients with and without heart disease?
- Which model and preprocessing steps produce the best, robust results?


In [31]:
# importing libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df=pd.read_csv('heart_cleaned.csv')


In [32]:
#Choose inputs(X)and output(y)
X =df.drop(columns=['target','has_disease','target_label','sex_label','exang'])
y =df['has_disease']

In [33]:
print(X.dtypes)
print(X.isnull().sum().sum())

age              int64
sex              int64
cp               int64
trestbps         int64
chol             int64
fbs              int64
restecg          int64
thalach          int64
oldpeak        float64
slope            int64
ca               int64
thal             int64
exang_label        str
dtype: object
0


In [39]:
print(X.shape)
print(df.shape              )

(302, 13)
(302, 18)


In [34]:
X=df.drop(columns=['target','has_disease']).select_dtypes(include="number")

#### Explanation 

X is what the model sees about each patient and y is what it tries to predict.You must drop **target*, because its the answer in another form and leavng it in woukd be cheating. This is called dat leakage.

In [40]:
#Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )



#### Explanation 

80% of the patients train the model and 20% are kept hidden for testing. *random_state=42* makes your split repeatable and *stratify=y* keeps the disease share similar in both parts. 

Check *X_train.shape* and *X_test.shape*

In [41]:
#scale the features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


### Explanation 

Cholesterol is in the hundreds and sex is 0 or 1.Scaling puts every column on a similar range so the model can compare them fairly. Notice that the scaler learns only from the training data,then is applied to the test data.

In [42]:
#Training the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
predictions =model.predict(X_test_s)


### Explanation 

logistic regression gives each patient a probability between 0 and 1,and predicts "disease" if its above 0.5 *fit* is learning step.

In [43]:
#work out the accuracy of the model
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy of the logistic regression model: {accuracy:.2f}")


Accuracy of the logistic regression model: 0.82


### Explanation 

82%. Thats is roughly 50 correct out of 61 test patients so abot 11 were wrong.

In [44]:
#find the baseline accuracy of the model of the model by predicting the majority class
y_test.value_counts(normalize=True) 

has_disease
0    0.540984
1    0.459016
Name: proportion, dtype: float64

### Explanation 

 The test is small.Each patient is worth about 1.6% points so a different random split could easily give 72% or 82%. Treat 82% as "around 82%" not an exact figure. Accuracy hides the type of mistake.The 11 errors could be missed sick patients or false alarms, and in medicine that difference matters a lot.

In [45]:
# learned weights of the model
weights = pd.Series(model.coef_[0], index=X.columns).sort_values()
weights

cp         -0.738433
slope      -0.524645
thalach    -0.375614
restecg    -0.149370
age        -0.064003
fbs        -0.001250
chol        0.225846
trestbps    0.261779
exang       0.430675
oldpeak     0.458287
thal        0.550544
sex         0.790243
ca          1.090547
dtype: float64

### Explanation 

Positive weights push toward disease and negative weights push away

In [46]:
#probabilites behind the predictions
probs = model.predict_proba(X_test_s)[:, 1]  # Probabilities for the positive class 
pd.DataFrame({'probability': probs.round(2),
              'predicted': predictions,
                'actual': y_test.values}).head(10)

,probability,predicted,actual
0,0.20,0,0
1,0.37,0,0
2,0.02,0,0
3,0.02,0,0
4,0.28,0,0
5,0.02,0,0
6,0.13,0,0
7,0.28,0,0
8,0.72,1,1
9,0.32,0,0


### Confusion matrix 

Confusion matrix is a simple table used to measure how well a classification model is performing. It compares the predictions made by the model with the actual results and shows where the model was right or wrong. This helps you understand where the model is making mistakes so you can improve it. It breaks down the predictions into four categories:

True Positive (TP): The model correctly predicted a positive outcome i.e the actual outcome was positive.
True Negative (TN): The model correctly predicted a negative outcome i.e the actual outcome was negative.
False Positive (FP): The model incorrectly predicted a positive outcome i.e the actual outcome was negative. It is also known as a Type I error.
False Negative (FN): The model incorrectly predicted a negative outcome i.e the actual outcome was positive. It is also known as a Type II error.

In [47]:
from sklearn.metrics import confusion_matrix, classification_report         
print(confusion_matrix(y_test, predictions))

[[27  6]
 [ 5 23]]


In [49]:
print(classification_report(y_test, predictions, target_names=['No Disease', 'Has Disease']))

              precision    recall  f1-score   support

  No Disease       0.84      0.82      0.83        33
 Has Disease       0.79      0.82      0.81        28

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



#### Confusion matrix explanation

27 healthy patients correctly cleared (TN).
6 healthy patients wrongly flagged (FP).
5 sick patients missed (FN).
23 sick patients correctly caught (TP).

25+8+6+22=61
27+23=50
50/61=0.82 is the accuracy 

#### Where the report numbers came from 

- Sensitivity (recall for has disease) : 23/28=0.82. The model catches about 4 in 5 sick.
- Specificity (recall for no disease): 27/33=0.82. It correctly clears about 3 in 4  healthy patients.
- Precision for has disease (0.79):of the 29 patients it flagged ,23 really were sick.